In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

DEEP LEARNING -- ASSIGNMENT 1
===

*Purpose of this assignment is to predict the ´charges´ column of the insurance.csv file through a Neural Network model.

Optuna, Optimizers, (???)

In [ ]:
#NECESSARY IMPORTS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
 
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
 
import optuna
from optuna.samplers import TPESampler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
# SEED FOR REPRODUCIBILITY
# so we get the same results every time we run the notebook
SEED = 42

# !!!
# Set the random seed for both PyTorch and NumPy to ensure reproducibility of results.
torch.manual_seed(SEED)
np.random.seed(SEED)

# Check if an NVIDIA GPU (CUDA) is available. 
# If yes, we use 'cuda'; otherwise, we fall back to the 'cpu'.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Print the device choice so we know if the code is running on the GPU or CPU
print(f"Using device: {DEVICE}\n")

Using device: cpu



1. DATA PREPARATION AND UNDERSTANDING
---

Guides used (From Oficial PyTorch Documentation): [Data Tutorial](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html) || [Data Loading Tutorial](https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html)

In [ ]:
# 0. DATASET LOADING
# We define a function with a default file path ("insurance.csv") 
# and specify that it returns a pandas DataFrame.
def load_data(path: str = "insurance.csv") -> pd.DataFrame:
    # read the CSV file into a DataFrame
    df = pd.read_csv(path)
    # Check the size
    print(f"  Shape        : {df.shape}")
    # Check column names
    print(f"  Columns      : {df.columns.tolist()}")
    # Check missing values
    print(f"  Missing vals : {df.isnull().sum().sum()}")
    # Statistical summary of the 'charges' column (our target variable)
    # This shows the mean, min, max, and percentiles to spot outliers early.
    print(f"  Charges stats:\n{df['charges'].describe().to_string()}\n")
    return df

# Execute the function and store the result in a variable named 'df'
df = load_data()

  Shape        : (1338, 7)
  Columns      : ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']
  Missing vals : 0
  Charges stats:
count     1338.000000
mean     13270.422265
std      12110.011237
min       1121.873900
25%       4740.287150
50%       9382.033000
75%      16639.912515
max      63770.428010



In [ ]:
# 1.1 FEATURE ENGENEERING
# We use get_dummies to transform the 'region' column into multiple numeric columns.
# If 'region' has 4 categories (NE, NW, SE, SW), it will create 4 columns.
# 'drop_first=True' is used to avoid the "Dummy Variable Trap":
# It removes one column (e.g., 'region_northeast'). 
# If the remaining columns (NW, SE, SW) are all 0, the model knows the region MUST be NE.
# This prevents mathematical redundancy (multicollinearity) in your model.
data = pd.get_dummies(df, columns=['region'], drop_first=True)
# We manually map the strings to integers: Female becomes 0, Male becomes 1.
data['sex'] = data['sex'].map({'female': 0, 'male': 1})
# 'Binary Encoding' for smoking status:
# 'no' becomes 0 and 'yes' becomes 1.
data['smoker'] = data['smoker'].map({'no': 0, 'yes': 1})
# Display the first 5 rows of the new DataFrame to verify the changes.
data.head()


,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,False,False,True
1,18,1,33.770,1,0,1725.55230,False,True,False
2,28,1,33.000,3,0,4449.46200,False,True,False
3,33,1,22.705,0,0,21984.47061,True,False,False
4,32,1,28.880,0,0,3866.85520,True,False,False


In [ ]:
# 1.2 PREPROCESSING (NORMALIZATION, STANDARDIZATION, ETC.)
# Define which numeric columns need to be scaled. It is not necessary to scale the one-hot encoded 'region' columns because they are already in a 0/1 format.
# We include 'charges' here because it's our target variable and we want it on a similar scale as the features.
cols_a_escalar = ['age', 'bmi', 'children', 'charges']

# Initialize the StandardScaler.
# This follows the formula: z = (x - mean) / standard_deviation
# It centers the data around 0 with a standard deviation of 1.
scaler = StandardScaler()

# fit_transform does two things:
# 1. 'fit' calculates the average (mean) and variance of each column.
# 2. 'transform' actually applies the math to change the numbers.
data[cols_a_escalar] = scaler.fit_transform(data[cols_a_escalar])
# Ensure all data in the DataFrame is a float (decimal number).
# This prevents errors during PyTorch tensor conversion later.
data = data.astype(float)
data.head()

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,-1.484353,0.0,-0.020650,-0.371909,1.0,-0.368689,0.0,0.0,1.0
1,-1.484353,1.0,-0.020650,-0.371909,0.0,-0.368689,0.0,1.0,0.0
2,0.041042,1.0,-0.020650,1.534660,0.0,-0.368689,0.0,1.0,0.0
3,0.041042,1.0,-1.474832,-0.371909,0.0,-0.368689,1.0,0.0,0.0
4,0.041042,1.0,-0.020650,-0.371909,0.0,-0.368689,1.0,0.0,0.0


In [ ]:
# 1.3 DATA SPLITTING (TRAIN/VAL/TEST)
# X = every column except 'charges' (independent variable)
X = data.drop('charges', axis=1).values

# y = only the column 'charges' (dependent variable, what we want to predict)
y = data['charges'].values

# We split the data: 80% for training and 20% for testing.
# random_state=42 ensures the "random" split is the same every time you run the code.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# We print the number of rows in each set 
print(f"Muestras de entrenamiento: {len(X_train)}")
print(f"Muestras de prueba: {len(X_test)}")


# Convertimos a tensores de punto flotante (32 bits es el estándar en DL)
X_train = torch.tensor(X_train, dtype=torch.float32)
# .reshape(-1, 1) transforms 'y' from a flat list [1, 2, 3] 
# into a column vector [[1], [2], [3]], which PyTorch layers expect for regression.
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1) 

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

# TensorDataset wraps the features and targets into a single object
train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

# batch_size=32: Model studies 32 rows at a time before updating weights 
# data separated into 32 "groups" (batches)
batch_size = 32

# shuffle=True: Mixes the data so the model doesn't memorize 
# the specific order of the rows (improves generalization).train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

# No shuffle needed for testing; we just want to evaluate the fixed set.test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

Muestras de entrenamiento: 1070
Muestras de prueba: 268


2. NEURAL NETWORK DESIGN (Deep, Shallow...?)
---

In [ ]:
# 2.1 LAYERS AND UNITS

In [ ]:
# 2.2 ACTIVATION FUNCTIONS

3. TRAINING METHODOLOGY
---

In [ ]:
# 3.1 LOSS FUNCTION (Regression --> MSE?)

In [ ]:
# 3.2 OPTIMIZER (SGD, ADAM, ..., ?)

In [ ]:
# 3.3 INITIALIZATION (He Initialization?)

4. PERFORMANCE EVALUATION
---